# Metrics for time-series forecasting

_Metrics & Evaluation — measuring models, data & statistics_

**Score a forecast by how far it misses over time — and make sure it beats just repeating the last value.**

A forecast produces a number $\hat y_t$ for each future time $t$; the truth turns out to be $y_t$. The error at time $t$ is $e_t = y_t - \hat y_t$ (positive means we under-forecast). Every metric here is just a different honest way to summarize the whole sequence of errors $e_1, e_2, \dots, e_h$ into one number.

       The single most important idea: a number is only meaningful relative to a baseline. The baseline for forecasting is the naive forecast — "tomorrow equals today" (last value), or for seasonal data "this month equals the same month last year". If your fancy model cannot beat that, it has learned nothing. Scaled metrics (MASE, RMSSE, Theil's U) bake the baseline right into the score, so below 1 = better than naive, above 1 = worse.

---

This notebook is a practice scaffold for the **Metrics for time-series forecasting** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — scikit-learn

### Step 1 — Build a realistic series and split it by time

We synthesize a monthly series with three honest ingredients: a rising **trend**, a repeating **yearly season**, and random **noise**. The golden rule for time series is to split by time, never randomly — so the last 12 months become the held-out test horizon and everything before it is the training history.

In [ ]:
import numpy as np

# --- a real-shaped monthly series: trend + yearly season + noise ---
rng = np.random.default_rng(7)
t = np.arange(60)
y = (100 + 1.8*t
     + 18*np.sin(2*np.pi*(t % 12)/12) + 8*np.cos(2*np.pi*(t % 12)/6)
     + rng.normal(0, 4, 60)).round(1)

# SPLIT BY TIME, never randomly: last 12 months are the test horizon
m = 12
train, test = y[:48], y[48:]

### Step 2 — Make a naive baseline and a trend-aware forecast

The **seasonal-naive** baseline simply replays the most recent full season — "this month equals the same month a year ago". Our actual model takes that same season and lifts it by the fitted yearly trend, so we can later ask the only question that matters: does the model beat naive?

In [ ]:
# seasonal-naive baseline forecast: same month one year earlier
naive_fc = train[-m:]

# a simple model: shift last season up by the fitted trend over a year
slope = np.polyfit(np.arange(48), train, 1)[0]
model_fc = (naive_fc + slope * m).round(1)

### Step 3 — Score the point forecast several ways

Each metric summarizes the sequence of errors differently. **MAE** is the average absolute miss, **RMSE** squares errors so big misses hurt more, and **MAPE / sMAPE / WAPE** express the error as a percentage (sMAPE symmetrizes the denominator; WAPE pools the totals so individual near-zero actuals can't blow it up).

In [ ]:
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error,
                             mean_absolute_percentage_error)

# --- point-forecast metrics via sklearn.metrics ---
mae  = mean_absolute_error(test, model_fc)
rmse = np.sqrt(mean_squared_error(test, model_fc))
mape = mean_absolute_percentage_error(test, model_fc) * 100        # %

# sMAPE and WAPE are simple to write out:
smape = np.mean(np.abs(test-model_fc)/((np.abs(test)+np.abs(model_fc))/2)) * 100
wape  = np.sum(np.abs(test-model_fc)) / np.sum(np.abs(test)) * 100

print(f"MAE  {mae:.2f}  RMSE {rmse:.2f}  MAPE {mape:.2f}%  sMAPE {smape:.2f}%  WAPE {wape:.2f}%")

### Step 4 — Scale by the baseline, then score the quantiles

**MASE** and **RMSSE** divide the error by the in-sample seasonal-naive error, so a value below 1 means you beat naive and above 1 means you lost to it — baking the baseline straight into the score. The **pinball (quantile) loss** then scores a single quantile of the forecast distribution; for the median (q=0.5) it reduces to half the MAE.

In [ ]:
# --- MASE from scratch: divide MAE by the in-sample seasonal-naive MAE ---
def mase(y_train, y_true, y_pred, season):
    scale = np.mean(np.abs(y_train[season:] - y_train[:-season]))  # baseline error
    return mean_absolute_error(y_true, y_pred) / scale

# RMSSE is the same idea with squared errors under a root:
def rmsse(y_train, y_true, y_pred, season):
    scale = np.mean((y_train[season:] - y_train[:-season])**2)
    return np.sqrt(np.mean((y_true - y_pred)**2) / scale)

print(f"MASE  model={mase(train,test,model_fc,m):.3f}   "      # < 1 beats naive
      f"naive={mase(train,test,naive_fc,m):.3f}")              # ~ 1 by definition
print(f"RMSSE model={rmsse(train,test,model_fc,m):.3f}")       # the M5 metric

# --- pinball / quantile loss from scratch (here for the median, q=0.5) ---
def pinball_loss(y_true, y_pred_q, q):
    d = y_true - y_pred_q
    return np.mean(np.maximum(q*d, (q-1)*d))

print(f"pinball(q=0.5)={pinball_loss(test, model_fc, 0.5):.3f}")

# sklearn also ships this directly:
from sklearn.metrics import mean_pinball_loss
print(f"sklearn pinball(q=0.9)={mean_pinball_loss(test, model_fc, alpha=0.9):.3f}")

# --- residual diagnostics with statsmodels ---
# from statsmodels.stats.diagnostic import acorr_ljungbox
# from statsmodels.stats.stattools import durbin_watson
# resid = test - model_fc
# print(acorr_ljungbox(resid, lags=[6]))   # small p-value => leftover autocorrelation
# print(durbin_watson(resid))              # ~2 = no autocorrelation, ~0 = positive

# --- probabilistic scoring (CRPS) with properscoring / scoringrules ---
# import properscoring as ps
# crps = ps.crps_ensemble(observation=test, forecasts=sample_paths).mean()
# (scoringrules.crps_normal(mu, sigma, y) for a parametric forecast)

## Visualize the data & results

_On a real-shaped monthly series split by time, does a trend-aware seasonal model actually beat the naive last-season baseline?_

### Step 1 — Rebuild the same series, split, and forecasts

This block re-creates the identical monthly series and the time-based split, then forms the seasonal-naive baseline and the trend-aware model forecast again, so the visualization is self-contained.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# real-shaped monthly series: trend + yearly season + noise
rng = np.random.default_rng(7)
t = np.arange(60)
y = (100 + 1.8*t
     + 18*np.sin(2*np.pi*(t % 12)/12) + 8*np.cos(2*np.pi*(t % 12)/6)
     + rng.normal(0, 4, 60)).round(1)

m = 12
train, test = y[:48], y[48:]            # SPLIT BY TIME (never randomly)

naive_fc = train[-m:]                    # seasonal-naive baseline
slope = np.polyfit(np.arange(48), train, 1)[0]
model_fc = (naive_fc + slope*m).round(1) # trend-aware seasonal model

### Step 2 — Compare model vs naive on the same ruler

We print the actual test values next to both forecasts, then score each with MASE and RMSE. Because MASE is scaled by the naive baseline, reading the two MASE numbers side by side directly answers whether the trend-aware model earns its keep.

In [ ]:
def mase(y_train, y_true, y_pred, season):
    scale = np.mean(np.abs(y_train[season:] - y_train[:-season]))
    return mean_absolute_error(y_true, y_pred) / scale

print("months 49..60:", list(range(49, 61)))
print("actual:", test.tolist())
print("model :", model_fc.tolist())
print("naive :", naive_fc.tolist())
print("MASE model:", round(mase(train, test, model_fc, m), 3))
print("MASE naive:", round(mase(train, test, naive_fc, m), 3))
print("RMSE model:", round(np.sqrt(mean_squared_error(test, model_fc)), 2))
print("RMSE naive:", round(np.sqrt(mean_squared_error(test, naive_fc)), 2))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You forecast next year's monthly sales for 500 different products at once and want a single leaderboard. Some products sell ~10 units a month, others ~10,000. A colleague suggests averaging each product's RMSE. Why is that a bad idea, and what should you use instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- See the units problem. — _RMSE is in the series' own units, so a 10,000-unit product can have an RMSE of 800 while a 10-unit product's RMSE is 3. Averaging them lets the big sellers dominate the leaderboard regardless of forecast skill._
- Switch to a scale-free metric. — _MASE and RMSSE divide each series' error by that series' own naive-baseline error, producing a unitless number where 1 = ties the baseline. Now every product is on the same ruler._
- Read the value. — _Average MASE (or RMSSE) below 1 means the model beats naive across the portfolio; above 1 means it doesn't. The M5 competition ranked exactly this way with RMSSE._

**Answer:** Averaging RMSE across series of wildly different magnitudes lets the high-volume products swamp the score, so the leaderboard measures size, not skill. Use MASE or RMSSE: each divides a series' error by its own seasonal-naive baseline error, giving a unitless score where below 1 beats naive. Average that across the 500 products for a fair leaderboard — this is the M5 approach.

</details>

**Problem 2.** A demand model reports MAPE = 4% and everyone is thrilled — until you notice several weeks had actual demand of 0 (stockouts). What's wrong, and how should you re-score?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the divide-by-near-zero. — _MAPE divides each error by the actual $|y_t|$. When $y_t=0$ the term is infinite (or undefined); when it is tiny, the term explodes and a single point can dominate or quietly get dropped, making the headline 4% untrustworthy._
- Pick a robust percentage. — _WAPE divides total error by total actual, so individual zeros don't blow up — and it stays defined as long as the totals aren't all zero._
- Or go baseline-relative. — _MASE scales by the seasonal-naive in-sample error, which also never divides by a single near-zero point and additionally tells you whether you beat the baseline._

**Answer:** MAPE divides by each actual value, so the zero-demand weeks make terms blow up or be silently dropped — the 4% is meaningless. Re-score with WAPE (total error ÷ total actual, robust to zeros) or MASE (scaled by the naive baseline error). Both avoid dividing by a single near-zero point, and MASE also answers the question that matters: did the model beat just-repeat-last-season?

</details>

**Problem 3.** Your forecast's point accuracy (RMSE) is good, but the residuals (actual minus forecast) clearly drift up and down in a slow wave over time. A Durbin-Watson statistic comes back at 0.4 and a Ljung-Box test gives p = 0.001. What do these tell you, and what should you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Interpret Durbin-Watson. — _Durbin-Watson runs 0 to 4, with ~2 meaning no residual autocorrelation. 0.4 is near 0, signalling strong positive autocorrelation — consecutive residuals are similar, so the model is systematically missing a moving pattern._
- Interpret Ljung-Box. — _Ljung-Box tests whether residual autocorrelation is jointly zero; p = 0.001 (p < 0.05) rejects that — there is leftover structure the model failed to capture._
- Fix the model, not the metric. — _Autocorrelated residuals mean an un-modeled trend or seasonality. Add a seasonal term, a trend, or lagged features (or move to a model like SARIMA) until the residuals look like uncorrelated noise (Durbin-Watson near 2, Ljung-Box p large)._

**Answer:** Both diagnostics say the residuals still contain a pattern: Durbin-Watson of 0.4 (far below 2) flags strong positive autocorrelation, and Ljung-Box p = 0.001 rejects "no autocorrelation". Good RMSE is hiding an un-modeled trend/season. Don't ship it — add the missing seasonal or trend component (e.g. move to SARIMA or add lagged features) until the residuals are uncorrelated noise (Durbin-Watson ≈ 2, Ljung-Box p large).

</details>